## Comparison of coordinate systems and resolutions for dehydration calcs:

In [ ]:
import sys, os, shutil
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import numpy as np
import math
import matplotlib.pyplot as pl
import shapely.geometry as geo
import pyvista as pv
import pathlib
import hashlib
import zipfile
import requests
from dataclasses import dataclass
import itertools

In [ ]:
import fenics_sz.utils
from fenics_sz.sz_problems.sz_params import allsz_params
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.fluid_release.perple_x_class import PerpleXGrid
from fenics_sz.fluid_release.slab_dehydration_class import SlabDehydration, SlabMesh            #call for slab-orthogonal slab mesh
from fenics_sz.fluid_release.vertical_slab_dehyrdation_class import SlabDehydrationVertical     #call for near-orthogonal slab mesh
#FIXME above import has a typo in the name

#### Choose a subduction zone to analyze

In [ ]:
name = "03_British_Columbia"
resscale = 5.0

In [ ]:
szdict = allsz_params[name]
print("{}:".format(name))
print("{:<20} {:<10}".format('Key','Value'))
print("-"*85)
for k, v in allsz_params[name].items():
    if v is not None: print("{:<20} {}".format(k, v))

slab = create_slab(szdict['xs'], szdict['ys'], resscale, szdict['lc_depth'])
_ = plot_slab(slab)

#### Load pyvista grids from zenodo

In [ ]:
zipfilename = pathlib.Path(os.path.join(basedir, os.path.pardir, os.path.pardir, "data", "vankeken_wilson_peps_2023_TF_lowres_minimal.zip"))
if not zipfilename.is_file():
    zipfileurl = 'https://zenodo.org/records/13234021/files/vankeken_wilson_peps_2023_TF_lowres_minimal.zip'
    r = requests.get(zipfileurl, allow_redirects=True)
    open(zipfilename, 'wb').write(r.content)
assert hashlib.md5(open(zipfilename, 'rb').read()).hexdigest() == 'a8eca6220f9bee091e41a680d502fe0d'

In [ ]:
tffilename = os.path.join('vankeken_wilson_peps_2023_TF_lowres_minimal', 'sz_suite_td', szdict['dirname']+'_minres_2.00_cfl_2.00.vtu')
tffilepath = os.path.join(basedir, os.path.pardir, os.path.pardir, 'data')
with zipfile.ZipFile(zipfilename, 'r') as z:
    z.extract(tffilename, path=tffilepath)
tfgrid = pv.get_reader(os.path.join(tffilepath, tffilename)).read() # the pv grid that is passed into our functions; lets us avoid solving the pdes again

In [ ]:
dmm_thickness = 2.0

tres = 2
sres = 20

# negative number implies below slab, positive implies above it
layer_thicknesses = [
                #  2.0,            # above slab mantle
                 -szdict['z15'], # sediments
                 -0.3,           # upper volcanics
                 -0.3,           # lower volcanics
                 -1.4,           # dikes
                 -5.0,           # gabbro
                 -dmm_thickness  # subslab mantle
                ]

csv_path = os.path.join(os.pardir, os.pardir, 'data', 'perple_x_v7.1.9', 'abers_25')
layer_h2os = [
    # PerpleXGrid(csv_file=os.path.join(csv_path, 'DMMdry_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, szdict['sed_type']+'_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'upvolc_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'lovolc_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'dike_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'gabbro_25_h2o.csv')),
    PerpleXGrid(csv_file=os.path.join(csv_path, 'DMMdamp_25_h2o.csv'))
]

layer_tres = [
    None,
    None,
    None,
    None,
    1.4,
]

In [ ]:
# array for varying slab-tangent resolutions:

sres_list = [1, 2, 3, 5, 7.5, 10, 15, 20, 50, 100]

#### Call the SlabDehydration Class for different resolutions

In [ ]:
conservation_errs_orth = []
conservation_errs_near_orth = []
cumulative_water_loss_orth = []
cumulative_water_loss_near_orth = []


for sres_test in (sres_list):
    
    testslab = SlabDehydration(sres_test, tres, layer_thicknesses, layer_h2os, layer_tres=None,
                           slab=slab, Tgrid=tfgrid, 
                           Tname='Temperature::PotentialTemperature', 
                           coast_distance=szdict['coast_distance'], 
                           sztype=szdict['sztype'], lc_depth=szdict['lc_depth'], trench_length=szdict['trench_length'], Vs=szdict['Vs'])
    conservation_errs_orth.append(sum([errors.sum() for errors in testslab.conservation_errors]))
    cumulative_water_loss_orth.append((testslab.total_cumulative_H2O_losses/1000.0)[-1])

    print("sres ", sres_test, " orthogonal run complete")

    testslab = SlabDehydrationVertical(sres_test, tres, layer_thicknesses, layer_h2os, layer_tres=None,
                            slab=slab, Tgrid=tfgrid, 
                            Tname='Temperature::PotentialTemperature', 
                            coast_distance=szdict['coast_distance'], 
                            sztype=szdict['sztype'], lc_depth=szdict['lc_depth'], trench_length=szdict['trench_length'], Vs=szdict['Vs'])
    conservation_errs_near_orth.append(sum([errors.sum() for errors in testslab.conservation_errors]))
    cumulative_water_loss_near_orth.append((testslab.total_cumulative_H2O_losses/1000.0)[-1])
    print("sres ", sres_test, " near orthogonal run complete")


In [ ]:
print(conservation_errs_orth)
print(conservation_errs_near_orth)

#### Plot the different conservation errors as a function of sres

In [ ]:
fig, ax = pl.subplots()
ax.plot(sres_list, conservation_errs_orth,  marker = "^", label = "orthogonal")
ax.plot(sres_list, conservation_errs_near_orth, marker = ".", linestyle="dotted", label="vertical")
ax.set_xlabel("sres")
ax.set_ylabel("conservation error")
ax.set_title("Conservation Errors at Resolutions")
ax.legend()

In [ ]:
fig, ax = pl.subplots()
ax.plot(sres_list, cumulative_water_loss_orth,  marker = "^", label = "orthogonal")
ax.plot(sres_list, cumulative_water_loss_near_orth, marker = ".", linestyle="dotted", label="vertical")
ax.set_xlabel("sres")
ax.set_ylabel("cumulative water loss")
ax.set_title("Cumulative Water Loss at Resolutions")
ax.legend()